In [ ]:
"""
Load L9 Benchmark Simulations into AnnData

Produces subset (HVG) and full-gene AnnData objects for all 27 datasets
(9 scenarios × 3 replicates) from the L9 orthogonal array benchmark.

FOLDER STRUCTURE EXPECTED:
  l9_benchmark/
    S1/
      rep1/  ← counts.mtx, genes.txt, cells.txt, metadata.csv
      rep2/
      rep3/
    S2/ ...
    S9/
      rep3/

OUTPUT (saved inside each rep folder):
  l9_benchmark/S1/rep1/S1_rep1_anndata_subset.h5ad
  l9_benchmark/S1/rep1/S1_rep1_anndata_full.h5ad
"""

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.io import mmread
from pathlib import Path
import gc

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

base_path = Path('l9_benchmark_sets')

# ============================================================================
# DISCOVER ALL REPLICATE DIRECTORIES
# ============================================================================
#
# The L9 structure is two levels deep: base_path/scenario/replicate/
# We use rglob to find all counts.mtx files anywhere under base_path,
# then take their parent directories — these are the rep-level folders.

rep_dirs = sorted([p.parent for p in base_path.rglob('counts.mtx')])

print()
print('=' * 68)
print('  L9 Benchmark — Load to AnnData')
print('=' * 68)
print(f'Base directory : {base_path}')
print(f'Datasets found : {len(rep_dirs)}  (expect 27)')
print()

In [ ]:
# ============================================================================
# PROCESSING LOOP
# ============================================================================

for rep_dir in rep_dirs:

    # Derive human-readable name from path: l9_benchmark/S1/rep1 -> "S1_rep1"
    sim_name = f"{rep_dir.parent.name}_{rep_dir.name}"  # e.g. "S1_rep1"

    print('=' * 68)
    print(f'Processing: {sim_name}')
    print(f'Path:       {rep_dir}')
    print('=' * 68)



    # --- 1. LOAD DATA -------------------------------------------------------

    # Load counts and transpose to (cells x genes) format
    counts   = mmread(rep_dir / 'counts.mtx').T.tocsr()
    genes    = pd.read_csv(rep_dir / 'genes.txt', header=None)[0].values
    cells    = pd.read_csv(rep_dir / 'cells.txt', header=None)[0].values
    metadata = pd.read_csv(rep_dir / 'metadata.csv', index_col=0)

    # Ground truth labels — consistent column name used by benchmarking framework
    metadata['Ground_Truth_Celltype'] = metadata['cell_type']
    unique_types                      = sorted(metadata['cell_type'].unique())
    type_to_cluster                   = {t: i+1 for i, t in enumerate(unique_types)}
    metadata['Ground_Truth_Cluster']  = metadata['cell_type'].map(type_to_cluster)

    # Create AnnData object
    adata          = ad.AnnData(X=counts, obs=metadata)
    adata.var_names = genes
    adata.obs_names = cells



    # --- 2. GLOBAL PREPROCESSING & RAW LAYERS ------------------------------

    # Filter cells and genes — sequential order matches Seurat pipeline
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)

    print(f'  Cells after QC : {adata.n_obs}')
    print(f'  Genes after QC : {adata.n_vars}')

    # Layer 1: Raw integer counts (required by scVI, CellAssign, etc.)
    adata.layers['counts'] = adata.X.copy()

    # Select HVGs using seurat_v3 flavor (requires raw counts — must run before log)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')

    # Normalize and log-transform
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Layer 2: Log-normalized counts (used by LLM-based and marker-based tools)
    adata.layers['log1p'] = adata.X.copy()



    # --- 3. FORK A: SUBSETTED (Standard Benchmark — HVGs only) ------------

    print(f'  → Creating Subsetted version (2000 HVGs)...')

    adata_sub = adata[:, adata.var.highly_variable].copy()
    sc.pp.scale(adata_sub, max_value=10)
    sc.tl.pca(adata_sub, n_comps=50, mask_var=None)

    subset_file = rep_dir / f"{sim_name}_anndata_subset.h5ad"
    adata_sub.write(subset_file)
    del adata_sub



    # --- 4. FORK B: FULL OBJECT (All genes — stress test) -----------------

    print(f'  → Creating Full version (all genes, scaled + PCA)...')

    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, mask_var=None)

    full_file = rep_dir / f"{sim_name}_anndata_full.h5ad"
    adata.write(full_file)



    # --- 5. CLEAN UP -------------------------------------------------------

    del adata, counts, metadata
    gc.collect()

    print(f'  ✓ Saved: {sim_name}\n')


print('=' * 68)
print(f'  ✓ All {len(rep_dirs)} datasets processed')
print(f'  Output: {len(rep_dirs) * 2} AnnData objects (subset + full)')
print('=' * 68)